# 04 — Dataset Tracking dengan DVC
**Proyek:** Klasifikasi Tingkat Risiko Stroke Berdasarkan Data Klinis dan Gaya Hidup Menggunakan Algoritma TabNet dengan Interpretasi Attention Mechanism

**Tahap:** Dataset Versioning & Tracking

**Tujuan:**
1. Menginisialisasi **DVC** sebagai tool untuk versioning dataset.
2. Melacak file dataset raw & processed agar reprodusibel.
3. Membuat pipeline `dvc.yaml` untuk otomatisasi tahapan preprocessing.
4. Menyimpan metadata versi (hash, timestamp, ukuran, changelog).
5. Menyediakan mekanisme rollback bila dataset berubah.

> **Mengapa DVC?**
> DVC memperlakukan dataset seperti Git memperlakukan source code. Kita dapat melakukan `git checkout` ke versi lama dan otomatis mendapatkan versi dataset yang konsisten — sangat penting untuk riset yang harus bisa direproduksi.


## 1. Setup

In [2]:
import hashlib
import json
import subprocess
from datetime import datetime
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
METADATA_DIR = PROJECT_ROOT / "metadata"
METADATA_DIR.mkdir(parents=True, exist_ok=True)

def run(cmd: str):
    """Run shell command dan return stdout."""
    print(f"$ {cmd}")
    result = subprocess.run(cmd, shell=True, cwd=PROJECT_ROOT,
                            capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0 and result.stderr:
        print("STDERR:", result.stderr)
    return result


## 2. Inisialisasi Git + DVC

In [ ]:
# Inisialisasi Git (jika belum ada repository)
if not (PROJECT_ROOT / ".git").exists():
    run("git init")
    run("git config user.email 'ridho@example.com'")
    run("git config user.name 'Ridho'")
else:
    print("Repository Git sudah ada.")


In [3]:
# Inisialisasi DVC
if not (PROJECT_ROOT / ".dvc").exists():
    run("dvc init")
else:
    print("DVC sudah terinisialisasi.")


DVC sudah terinisialisasi.


## 3. Menambahkan Dataset Raw ke Tracking DVC

In [5]:
# dvc add memindahkan file ke cache dan membuat .dvc pointer yang di-commit ke Git
run("dvc add data/raw/healthcare-dataset-stroke-data.csv")

# Tampilkan file pointer yang dibuat
pointer = PROJECT_ROOT / "data/raw/healthcare-dataset-stroke-data.csv.dvc"
if pointer.exists():
    print("\nIsi pointer DVC:")
    print(pointer.read_text())


$ dvc add data/raw/healthcare-dataset-stroke-data.csv
STDERR: /bin/sh: dvc: command not found



File `.dvc` inilah yang akan di-commit ke Git (berisi hash MD5 & ukuran file). File aktual dataset-nya ada di `.dvc/cache/` dan sebenarnya di-*ignore* oleh Git.

## 4. Membuat Pipeline `dvc.yaml`

In [4]:
dvc_yaml = '''stages:
  preprocess:
    cmd: jupyter nbconvert --to notebook --execute notebooks/03_data_preprocessing.ipynb --output 03_data_preprocessing.ipynb
    deps:
      - data/raw/healthcare-dataset-stroke-data.csv
      - notebooks/03_data_preprocessing.ipynb
      - params.yaml
    params:
      - dataset
      - preprocessing
    outs:
      - data/processed/train.csv
      - data/processed/val.csv
      - data/processed/test.csv
      - data/processed/tabnet_ready.npz
      - data/processed/scaler.joblib
      - data/processed/encoders.json
      - data/processed/tabnet_meta.json
    metrics:
      - metadata/03_preprocessing.json:
          cache: false
'''
(PROJECT_ROOT / "dvc.yaml").write_text(dvc_yaml)
print((PROJECT_ROOT / "dvc.yaml").read_text())


stages:
  preprocess:
    cmd: jupyter nbconvert --to notebook --execute notebooks/03_data_preprocessing.ipynb --output 03_data_preprocessing.ipynb
    deps:
      - data/raw/healthcare-dataset-stroke-data.csv
      - notebooks/03_data_preprocessing.ipynb
      - params.yaml
    params:
      - dataset
      - preprocessing
    outs:
      - data/processed/train.csv
      - data/processed/val.csv
      - data/processed/test.csv
      - data/processed/tabnet_ready.npz
      - data/processed/scaler.joblib
      - data/processed/encoders.json
      - data/processed/tabnet_meta.json
    metrics:
      - metadata/03_preprocessing.json:
          cache: false



### Penjelasan pipeline
- **deps**: file yang jika berubah akan memicu ulang stage.
- **params**: section di `params.yaml` yang jadi dependensi — jika diubah, stage re-run.
- **outs**: file output yang di-track DVC.
- **metrics**: file JSON untuk melacak hasil numerik (bisa dibandingkan antar run dengan `dvc metrics show`).

## 5. Mencatat Metadata Versi Dataset (Manual Backup)

In [ ]:
def file_sha256(path: Path, chunk: int = 1 << 16) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

def snapshot_directory(directory: Path) -> list:
    rows = []
    if not directory.exists():
        return rows
    for p in sorted(directory.rglob("*")):
        if p.is_file():
            rows.append({
                "relative_path": str(p.relative_to(PROJECT_ROOT)),
                "size_bytes": p.stat().st_size,
                "sha256": file_sha256(p),
            })
    return rows

dataset_version = {
    "version": "v0.1.0",
    "tag": "initial-preprocessing",
    "created_at": datetime.utcnow().isoformat() + "Z",
    "author": "Ridho",
    "changelog": [
        "Import dataset raw dari Kaggle (fedesoriano).",
        "Imputasi BMI dengan median.",
        "Hapus 1 baris gender=Other.",
        "IQR capping pada fitur numerik.",
        "Label encoding kategorikal + penyimpanan mapping.",
        "Split stratified 70/15/15.",
        "StandardScaler fit-only-on-train.",
        "SMOTE sampling_strategy=0.5 pada training set.",
    ],
    "raw": snapshot_directory(PROJECT_ROOT / "data" / "raw"),
    "processed": snapshot_directory(PROJECT_ROOT / "data" / "processed"),
}

version_path = METADATA_DIR / "dataset_versions.json"
# Append jika sudah ada
if version_path.exists():
    history = json.loads(version_path.read_text())
else:
    history = []
history.append(dataset_version)
version_path.write_text(json.dumps(history, indent=2))
print(f"Ditulis {len(history)} versi → {version_path}")
print(json.dumps(dataset_version, indent=2)[:1500], "...")


Kita menyimpan **metadata manual** sebagai *safety net* tambahan di luar DVC. Ini berguna sebagai jejak *human-readable* untuk laporan skripsi atau audit.

## 6. Git Commit untuk Membekukan Versi

In [ ]:
# Tambahkan pointer DVC + config ke Git (bukan data mentah!)
run("git add .gitignore .dvc .dvcignore data/raw/*.dvc dvc.yaml params.yaml")
run("git add notebooks/ metadata/ requirements.txt README.md")
run("git status --short")


In [ ]:
# Commit
run('git commit -m "feat(data): initial raw + preprocessed dataset tracked with DVC v0.1.0"')
run("git log --oneline -5")


> Setelah commit, Git menyimpan *pointer* ke versi dataset. Kapan pun kita checkout commit ini, `dvc checkout` akan mengembalikan dataset ke keadaan yang sama bit demi bit.

## 7. Verifikasi Status Tracking

In [ ]:
run("dvc status")


In [ ]:
run("dvc dag")   # visualisasi pipeline dependency (jika pipeline ada)


## 8. Reprodusibilitas — Reproduce Pipeline

In [ ]:
# Tidak perlu dijalankan sekarang; panggil saat ingin menjalankan ulang pipeline
# run("dvc repro")
print("Untuk menjalankan ulang pipeline preprocessing:")
print("    dvc repro")
print()
print("Untuk kembali ke versi sebelumnya:")
print("    git checkout <commit-hash>")
print("    dvc checkout")


## 9. Ringkasan Strategi Tracking

| Lapisan | Tool | Fungsi |
|---|---|---|
| **Source code & config** | Git | Melacak notebook, params.yaml, requirements, README |
| **Dataset artefak** | DVC | Melacak file besar (CSV, NPZ, joblib, JSON) |
| **Metadata manual** | JSON di `metadata/` | Human-readable changelog & hash SHA-256 |
| **Pipeline** | `dvc.yaml` | Definisi stage + otomatisasi reproduce |
| **Parameter** | `params.yaml` | Single source of truth untuk konfigurasi |

### Alur kerja harian
1. Edit notebook / `params.yaml` → `dvc repro` → output diperbarui otomatis.
2. `dvc add` untuk dataset baru.
3. `git add` pointer `.dvc` + `git commit` → versi dibekukan.
4. `git push` + `dvc push` (jika menggunakan remote storage).

### Rollback
```bash
git log --oneline               # cari commit target
git checkout <hash>             # kode kembali
dvc checkout                    # dataset kembali
```

**Proyek siap untuk tahap selanjutnya →** membangun & melatih model **TabNet**.
